# Transformers: route support messages and inspect what the model learned

A support team receives short messages about billing, account access and delivery. Before automatically routing them, you need an input contract, a baseline, a clear evaluation split and a way to inspect wrong predictions. We will work through those decisions using Hugging Face Transformers and a small BERT classifier.

This notebook contains the explanations, dataset, training, evaluation, persistence and reference notes in one place. **The default project runs offline on CPU after installation.** Its model starts with random weights and learns from a small synthetic dataset. It is not a pretrained language model and does not demonstrate production language understanding. An optional section explains the separate pretrained workflow.

You will practice tokenization, special tokens, padding, attention masks, architecture versus weights, batching, loss, validation, inference, pipelines and saving. Before each calculation, predict the shapes and what the output would establish. Allow 1–2 hours for study; the small training run should take much less.

## 1. Keep the environment and the experiment explicit

Use the repository environment, or install `transformers==4.57.3`, `torch==2.8.0`, `scikit-learn==1.6.1`, pandas, numpy and matplotlib into your kernel. The version pin supports the repository's Python 3.9 setup; it is not a latest-version claim.

We use the PyTorch backend explicitly because this workspace also contains TensorFlow. Restart the kernel if Transformers was already imported with different backend settings. Generated files go under `outputs/transformers/`; rerunning replaces this tutorial's dataset, plots and saved model. No Hub authentication is needed.

In [ ]:
import os
os.environ['USE_TF'] = '0'
os.environ['USE_TORCH'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
from pathlib import Path
import copy, json, re, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import transformers
from transformers import BertTokenizerFast, BertConfig, BertForSequenceClassification
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay
from IPython.display import display

OUT = Path('outputs/transformers'); OUT.mkdir(parents=True, exist_ok=True)
SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED); torch.set_num_threads(2)
print({'python': platform.python_version(), 'transformers': transformers.__version__, 'torch': torch.__version__})

## 2. Define a label a human could assign consistently

One row is a message with one intended routing label, not a conversation or a customer. We generate 240 messages from 10 issue phrases per class and 8 surrounding templates. Every class has the same count. Training uses 5 templates, validation 1, and testing 2 unseen templates.

Splitting by template avoids exact template reuse across splits, but the issue phrases recur deliberately. This tests a limited ability to recognize familiar issues in different wrappers; it does not test unseen intent, multilingual text or new customers. Real support data needs deduplication and customer/conversation grouping before splitting.

Predict the row count and per-class counts before running. A ticket containing two requests would require a label policy or multilabel design beyond this exercise.

In [ ]:
issues = {
    'billing': ['invoice amount is wrong', 'refund has not arrived', 'charged twice for purchase',
                'payment was declined', 'billing address needs update', 'receipt is missing',
                'subscription fee increased', 'credit card payment failed', 'cancel recurring charge', 'tax on invoice is incorrect'],
    'account': ['password reset is failing', 'cannot login to account', 'change account email',
                'verification code is missing', 'unlock my account', 'update my profile name',
                'enable two factor authentication', 'delete my account', 'username is forgotten', 'sign in session expired'],
    'delivery': ['package has not arrived', 'tracking number is missing', 'delivery address is wrong',
                 'parcel arrived damaged', 'shipment is delayed', 'reschedule package delivery',
                 'courier cannot find address', 'order was delivered elsewhere', 'track my shipment', 'package lost in transit']
}
templates = [
    'please help because {issue}', 'i need assistance with {issue}', 'my request is that {issue}',
    'can you check why {issue}', 'hello team {issue}', 'could someone investigate {issue}',
    'today i noticed that {issue}', 'i am contacting support since {issue}'
]
labels = sorted(issues)
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for label, i in label2id.items()}
rows = []
for label, phrases in issues.items():
    for template_id, template in enumerate(templates):
        for phrase in phrases:
            rows.append({'text': template.format(issue=phrase), 'label': label2id[label],
                         'template_id': template_id,
                         'split': 'train' if template_id < 5 else ('validation' if template_id == 5 else 'test')})
data = pd.DataFrame(rows)
assert len(data) == 240 and data.text.is_unique
train = data[data.split == 'train'].reset_index(drop=True)
valid = data[data.split == 'validation'].reset_index(drop=True)
test = data[data.split == 'test'].reset_index(drop=True)
assert set(train.template_id).isdisjoint(valid.template_id)
assert set(train.template_id).isdisjoint(test.template_id)
data.to_csv(OUT / 'synthetic_tickets.csv', index=False)
display(pd.crosstab(data.split, data.label).rename(columns=id2label))
display(train.sample(3, random_state=SEED))

## 3. Give a simpler model the same opportunity

Does this dataset require contextual attention at all? A TF-IDF linear classifier is a strong check when class-specific words recur. Fit its vocabulary on training text only. Record validation performance now and defer test comparison until the neural training protocol is fixed.

If a larger model cannot beat this baseline, complexity alone is not a reason to prefer it. Even if both perform very well, repeated synthetic phrases make the task easier than real support routing.

In [ ]:
baseline = make_pipeline(TfidfVectorizer(ngram_range=(1,2)), LogisticRegression(max_iter=1000, random_state=SEED))
baseline.fit(train.text, train.label)
print('Baseline validation macro F1:', f1_score(valid.label, baseline.predict(valid.text), average='macro'))

## 4. Inspect what survives tokenization

A model consumes integer token IDs, not strings. For an offline demonstration we construct a small BERT vocabulary from training words only. This is a teaching tokenizer, not the WordPiece vocabulary of a downloaded pretrained BERT.

`[CLS]` begins the sequence; `[SEP]` separates/ends it; `[PAD]` fills batch positions; `[UNK]` represents an unrecognized token; `[MASK]` is used in masked-language modeling. A tokenizer’s IDs must match its model’s embedding table. Replacing the tokenizer of a pretrained model with this one would break that contract.

Predict which words in the new wrappers become unknown. Unknown words do not become meaningful just because the network emits a confident score.

In [ ]:
words = sorted(set(re.findall(r'[a-z]+', ' '.join(train.text))))
vocab = ['[PAD]', '[UNK]', '[CLS]', '[SEP]', '[MASK]'] + words
(OUT / 'vocab.txt').write_text('\n'.join(vocab) + '\n')
tokenizer = BertTokenizerFast(vocab_file=str(OUT / 'vocab.txt'), do_lower_case=True, model_max_length=40)
example = tokenizer(valid.text.iloc[0])
display(pd.DataFrame({'id': example['input_ids'], 'token': tokenizer.convert_ids_to_tokens(example['input_ids'])}))

def unknown_rate(texts):
    ids = [i for text in texts for i in tokenizer(text, add_special_tokens=False)['input_ids']]
    return sum(i == tokenizer.unk_token_id for i in ids) / max(len(ids), 1)
print({name: round(unknown_rate(frame.text), 3) for name, frame in [('train',train), ('validation',valid), ('test',test)]})

## 5. Pad a batch without treating padding as evidence

A batch needs rectangular tensors. `padding=True` pads to the longest example in that batch; `truncation=True` enforces the maximum token count. The attention mask marks real positions with 1 and padding with 0. Token type IDs distinguish segments in BERT inputs; a single-message batch normally contains zeros.

Inspect a short and long example together. Predict the tensor shape and number of padded positions. Truncation can remove the very sentence that explains a ticket, so measure lengths before choosing a limit.

In [ ]:
MAX_LENGTH = 40
batch = tokenizer(['refund missing', train.text.iloc[0]], padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors='pt')
print({key: tuple(value.shape) for key, value in batch.items()})
print(batch['attention_mask'])
lengths = [len(tokenizer(t, add_special_tokens=True)['input_ids']) for t in data.text]
assert max(lengths) <= MAX_LENGTH
print('Maximum sequence length:', max(lengths))

def encode_frame(frame):
    encoded = tokenizer(frame.text.tolist(), padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors='pt')
    return encoded, torch.tensor(frame.label.to_numpy(), dtype=torch.long)
train_x, train_y = encode_frame(train)
valid_x, valid_y = encode_frame(valid)
test_x, test_y = encode_frame(test)

## 6. Separate a model architecture from learned knowledge

A configuration describes dimensions and behavior; it contains no learned language knowledge. Creating `BertForSequenceClassification(config)` initializes random weights. The classifier maps the encoded sequence to three logits. We use two small encoder layers, a hidden width of 32 and four attention heads so training is modest on CPU.

An encoder reads both left and right context. Decoder-only models are commonly used for next-token generation; encoder-decoder models map an input sequence to an output sequence. A routing classifier needs a classification head, not an arbitrary text-generation checkpoint.

The first forward pass should produce `(batch_size, 3)` logits. Supplying integer labels also requests cross-entropy loss. Raw logits are not probabilities.

In [ ]:
config = BertConfig(vocab_size=len(tokenizer), hidden_size=32, num_hidden_layers=2,
                    num_attention_heads=4, intermediate_size=64, max_position_embeddings=64,
                    num_labels=3, id2label=id2label, label2id=label2id,
                    pad_token_id=tokenizer.pad_token_id)
model = BertForSequenceClassification(config)
model.eval()
with torch.no_grad():
    initial = model(**{k:v[:4] for k,v in train_x.items()}, labels=train_y[:4])
assert tuple(initial.logits.shape) == (4,3)
print('Parameters:', sum(p.numel() for p in model.parameters()))
print('Initial loss:', initial.loss.item(), 'Uniform-reference loss:', np.log(3))

## 7. Train on messages and select weights with validation

Each epoch shuffles training rows, computes loss, backpropagates, clips the gradient norm and updates weights. `model.train()` enables training behavior such as dropout; `model.eval()` disables it. `torch.no_grad()` avoids building a gradient graph during evaluation.

We fix 18 epochs and a learning rate of 0.003 before examining the test results. Save the state with the lowest validation loss, not the final epoch by default. Validation is small, so a tiny change in its score is weak evidence.

Before execution, describe what declining training loss with increasing validation loss would suggest. A seed makes this CPU exercise repeatable in a fixed environment; it does not guarantee identical results on every platform.

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=0.003, weight_decay=0.01)
history = []; best_loss = float('inf'); best_state = None

def evaluate(encoded, targets):
    model.eval()
    with torch.no_grad():
        result = model(**encoded, labels=targets)
    return result.loss.item(), result.logits.argmax(dim=-1).numpy()

for epoch in range(18):
    model.train(); total = 0.0
    order = torch.randperm(len(train_y))
    for start in range(0, len(order), 24):
        idx = order[start:start+24]
        optimizer.zero_grad()
        result = model(**{k:v[idx] for k,v in train_x.items()}, labels=train_y[idx])
        result.loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total += result.loss.item() * len(idx)
    val_loss, val_prediction = evaluate(valid_x, valid_y)
    history.append({'epoch': epoch+1, 'train_loss': total/len(train_y), 'validation_loss': val_loss,
                    'validation_macro_f1': f1_score(valid.label, val_prediction, average='macro')})
    if val_loss < best_loss:
        best_loss = val_loss; best_state = copy.deepcopy(model.state_dict())
model.load_state_dict(best_state)
history = pd.DataFrame(history)
display(history.tail())
ax = history.plot(x='epoch', y=['train_loss','validation_loss'], figsize=(8,4), marker='o')
ax.set(ylabel='Cross-entropy loss', title='Small synthetic routing task')
plt.tight_layout(); plt.savefig(OUT / 'learning_curves.png'); plt.show()

## 8. Compare fixed models on the held-out templates

Macro F1 gives each class equal weight. The confusion matrix shows which routing mistakes occur, while accuracy summarizes the overall fraction correct. Compare the transformer with the TF-IDF baseline using the same held-out rows.

This comparison reports two prespecified workflows. Do not use the test score to keep changing the architecture. Perfect performance here would mainly show that the synthetic lexical patterns were learned, not that the system understands real tickets.

In [ ]:
test_loss, predictions = evaluate(test_x, test_y)
baseline_predictions = baseline.predict(test.text)
report = pd.DataFrame([
    {'model':'TF-IDF + logistic regression', 'accuracy':accuracy_score(test.label, baseline_predictions),
     'macro_f1':f1_score(test.label, baseline_predictions, average='macro')},
    {'model':'tiny BERT trained from scratch', 'accuracy':accuracy_score(test.label, predictions),
     'macro_f1':f1_score(test.label, predictions, average='macro')}
])
display(report)
print(classification_report(test.label, predictions, target_names=labels, zero_division=0))
ConfusionMatrixDisplay.from_predictions(test.label, predictions, display_labels=labels)
plt.tight_layout(); plt.show()
errors = test.assign(predicted=predictions).loc[lambda x: x.label != x.predicted]
display(errors.head(10))
if errors.empty:
    print('No errors on these easy held-out templates. Try new vocabulary, mixed intent and negation on a separate challenge set.')

## 9. Inspect contextual representations without claiming they are explanations

Request hidden states to inspect `(batch, tokens, hidden_width)`. Each token’s representation depends on its context. Pooling these values is possible, but a small classifier trained here is not a validated sentence-embedding model.

Attention weights, if requested from a compatible implementation, describe internal computation; they are not automatically a causal explanation of a prediction. For diagnosis, compare carefully designed input changes and labeled challenge cases.

In [ ]:
model.eval()
with torch.no_grad():
    representations = model.bert(**batch, output_hidden_states=True, return_dict=True)
print('Final hidden state:', tuple(representations.last_hidden_state.shape))
print('Embedding output plus encoder layers:', len(representations.hidden_states))
mask = batch['attention_mask'].unsqueeze(-1)
pooled = (representations.last_hidden_state * mask).sum(1) / mask.sum(1)
assert tuple(pooled.shape) == (2,32)

## 10. Save the tokenizer and label meanings with the weights

A model file alone is not the complete input contract. Save the tokenizer, configuration, class mapping and maximum length as well. `save_pretrained` names the serialization API; saving through it does not retroactively make our model pretrained on a language corpus.

Reload with `AutoTokenizer` and `AutoModelForSequenceClassification` using `local_files_only=True`. Check logits on fixed inputs before and after reloading. A successful load is weaker evidence than matching predictions.

In [ ]:
model_dir = OUT / 'routing_model'
model.eval()
model.save_pretrained(model_dir, safe_serialization=True)
tokenizer.save_pretrained(model_dir)
contract = {'task':'synthetic support routing', 'labels':labels, 'max_length':MAX_LENGTH,
            'initialization':'random; trained on synthetic tickets only', 'transformers':transformers.__version__}
(model_dir / 'input_contract.json').write_text(json.dumps(contract, indent=2))
loaded_tokenizer = AutoTokenizer.from_pretrained(model_dir, local_files_only=True)
loaded_model = AutoModelForSequenceClassification.from_pretrained(model_dir, local_files_only=True).eval()
with torch.no_grad():
    before = model(**test_x).logits
    after = loaded_model(**test_x).logits
assert torch.allclose(before, after, atol=1e-6)
print('Reloaded logits match.')

## 11. Use a pipeline while keeping the input assumptions visible

A pipeline combines preprocessing, model execution and postprocessing. We supply our local model explicitly, avoiding an implicit download of a default checkpoint. Its softmax score is not a guarantee of correctness or a calibrated probability of successful routing.

Try a familiar billing phrase and an unfamiliar question. The classifier must choose among its three labels even if none fits. An abstention threshold needs calibration and a labeled out-of-domain validation set; choosing 0.9 merely because it looks high is not enough.

In [ ]:
router = pipeline('text-classification', model=loaded_model, tokenizer=loaded_tokenizer, device=-1)
messages = ['please help because charged twice for purchase', 'what is the weather tomorrow']
results = router(messages, truncation=True, max_length=MAX_LENGTH, top_k=None)
for text, scores in zip(messages, results):
    print(text); print(scores)
    assert abs(sum(item['score'] for item in scores) - 1) < 1e-5
report.to_csv(OUT / 'test_metrics.csv', index=False)
history.to_csv(OUT / 'training_history.csv', index=False)
test.assign(predicted=predictions).to_csv(OUT / 'test_predictions.csv', index=False)

## 12. Move to a pretrained checkpoint deliberately

For a real text task, begin with a suitable pretrained checkpoint and its matching tokenizer. This optional example downloads a public sentiment model, which predicts sentiment rather than our routing labels. It is **off by default** and is not required by the project. Check its model card, license, intended language and dataset before using it; pin a reviewed Hub commit for reproducible work.

For routing fine-tuning, load an encoder classifier with your `num_labels`, inspect any newly initialized classification head, and train on properly split labeled tickets. `Trainer`/`TrainingArguments` can manage training and evaluation, but may need extra packages such as Accelerate. The explicit loop above makes the same boundaries visible without those dependencies.

In [ ]:
RUN_PRETRAINED_DEMO = False
if RUN_PRETRAINED_DEMO:
    sentiment = pipeline('sentiment-analysis', model='distilbert/distilbert-base-uncased-finetuned-sst-2-english', device=-1)
    print(sentiment('The support team resolved my issue quickly.'))
else:
    print('Optional pretrained download skipped; all project cells run offline.')

## 13. Choose the right abstraction for the next task

| Need | Relevant component | What to verify |
|---|---|---|
| Convert strings into model inputs | Tokenizer | Vocabulary, truncation, special tokens |
| Load an appropriate architecture | `AutoModelFor...` | Task head, config and label mapping |
| Quick inference | `pipeline` | Explicit checkpoint, device and batching |
| Repeatable fine-tuning | Trainer or a PyTorch loop | Splits, optimizer, evaluation and checkpoint policy |
| Generate text | Causal/sequence-to-sequence model and `generate` | Token budget, decoding settings, stop conditions |
| Reduce large-model memory | Supported quantization/PEFT tools | Hardware and model compatibility; separate evaluation |
| Share an artifact | `save_pretrained` plus metadata | Matching tokenizer and immutable version |

BERT classification is not a chat model. Decoder generation adds separate decisions: deterministic versus sampled decoding, maximum new tokens, context limits and prompt formatting. A model’s chat template, if any, belongs to its tokenizer and should not be invented arbitrarily.

On larger models, batching trades throughput against memory; dynamic padding can reduce wasted positions. CPU/GPU device placement and dtypes must be compatible. Keep this project on CPU first so device complexity does not conceal data mistakes.

## 14. Explain what the evidence supports

1. Replace an issue word with a synonym absent from the training vocabulary. Predict tokenization and compare the output. Explain why confidence may remain high even when information is lost.
2. Construct a labeled challenge set with negation, two intents and irrelevant text. Keep it separate from the template test and report performance by case type.
3. Compare the baseline and tiny BERT in accuracy, runtime and artifact size. State what would justify added complexity.
4. Before real fine-tuning, specify conversation grouping, annotation rules, a temporal holdout and an abstention evaluation.

| Observation | Investigate |
|---|---|
| Many `[UNK]` tokens | Vocabulary mismatch or domain/language shift |
| Sequence too long | Token-length distribution and lost tail information |
| Wrong label names | Saved `id2label` and annotation mapping |
| Training works but reload differs | Evaluation mode, tokenizer and saved weights |
| High test score but poor live routing | Repeated phrases, distribution shift and ambiguous labels |

Official references: [BERT architecture and classes](https://huggingface.co/docs/transformers/v4.57.1/en/model_doc/bert), [pipelines](https://huggingface.co/docs/transformers/v4.57.1/en/pipeline_tutorial), [tokenizer API](https://huggingface.co/docs/transformers/v4.57.1/en/main_classes/tokenizer), and [training](https://huggingface.co/docs/transformers/v4.57.1/en/training). Versioned documentation is close to the pinned patch release; this notebook prints its exact runtime version.

## Second project: Adapt a pretrained encoder under a small CPU budget

Compare a frozen pretrained encoder plus a newly trained classification head with the TF-IDF baseline on the support-ticket split already built above. Audit missing text and repeated imports, measure truncation, and learn what a pretrained model does and does not fix.

Run these cells in order. This extension uses its own `p2_` variables and writes only its named outputs under `outputs/project2/`. Predict each comparison before running it; the worked discussion after the code explains the result and its limits.

### Work through the transfer-learning boundary

The first project trained a tiny BERT from scratch. Here the encoder weights come from the pinned MiniLM snapshot used by the SentenceTransformers notebook. A new three-class head starts randomly; loading pretrained weights alone cannot route tickets. Freeze the encoder and train only the head for a bounded experiment. This keeps computation modest but limits how much the representation can adapt.

We retain the earlier phrase-group split so paraphrase wrappers cannot move an underlying issue across training and evaluation. Repeated exact imports are discarded; null text is rejected. The intentionally small authored dataset is a teaching fixture, not evidence of real support quality. Run all earlier cells first.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import time
from IPython.display import display
p2_out=Path('outputs/project2');p2_out.mkdir(parents=True,exist_ok=True)

from transformers import AutoTokenizer,AutoModelForSequenceClassification
from huggingface_hub import snapshot_download
import torch,copy
from sklearn.metrics import f1_score
p2_dir=Path('data/cache/all-MiniLM-L6-v2')
p2_revision='1110a243fdf4706b3f48f1d95db1a4f5529b4d41'
p2_marker=p2_dir/'notebook_revision.txt'
if not (p2_dir/'model.safetensors').exists() or not p2_marker.exists() or p2_marker.read_text().strip()!=p2_revision:
    snapshot_download('sentence-transformers/all-MiniLM-L6-v2',revision=p2_revision,local_dir=str(p2_dir),allow_patterns=['*.json','*.txt','model.safetensors','1_Pooling/*'])
    p2_marker.write_text(p2_revision)
p2_tokenizer=AutoTokenizer.from_pretrained(p2_dir,local_files_only=True)
torch.manual_seed(72);torch.set_num_threads(2)
p2_transfer=AutoModelForSequenceClassification.from_pretrained(p2_dir,num_labels=3,local_files_only=True)
for p in p2_transfer.base_model.parameters():p.requires_grad=False
p2_raw=pd.concat([train[['text','label']],train[['text','label']].iloc[:3],pd.DataFrame([{'text':None,'label':0}])],ignore_index=True)
p2_train=p2_raw.dropna(subset=['text','label']).drop_duplicates().copy()
assert p2_train.text.map(lambda x:isinstance(x,str) and bool(x.strip())).all()
p2_maxlength=64
p2_encoded=p2_tokenizer(p2_train.text.tolist(),padding=True,truncation=True,max_length=p2_maxlength,return_tensors='pt')
p2_targets=torch.tensor(p2_train.label.to_numpy(dtype='int64'))
p2_lengths=[len(p2_tokenizer(t,truncation=False)['input_ids']) for t in p2_train.text]
print({'raw':len(p2_raw),'accepted':len(p2_train),'truncated':sum(n>p2_maxlength for n in p2_lengths),'trainable_parameters':sum(p.numel() for p in p2_transfer.parameters() if p.requires_grad),'all_parameters':sum(p.numel() for p in p2_transfer.parameters())})

### Explain what a frozen encoder buys

Compute the encoder representations once in evaluation mode, then train the pooler and classifier on those fixed representations. No encoder gradient graph is retained. This is valid because the encoder is frozen and dropout is disabled there; it would be wrong for full fine-tuning, where representations must change each step.

MiniLM's sentence-embedding checkpoint does not train this classification pooler for our task. The pooler and classifier are trainable here, and validation chooses the best of 20 inexpensive head epochs. Compare with TF-IDF using the same held-out split. A larger model is not entitled to win.

In [ ]:
p2_start=time.perf_counter();p2_transfer.eval()
def p2_features(frame):
    encoded=p2_tokenizer(frame.text.tolist(),padding=True,truncation=True,max_length=p2_maxlength,return_tensors='pt')
    with torch.no_grad():return p2_transfer.bert.embeddings.word_embeddings.weight.new_tensor(p2_transfer.bert(**encoded).last_hidden_state[:,0].detach())
# Train a small head directly on cached CLS vectors; bypass the checkpoint's pooler.
p2_fx=p2_features(p2_train);p2_fv=p2_features(valid)
p2_head=torch.nn.Linear(p2_fx.shape[1],3);p2_opt=torch.optim.AdamW(p2_head.parameters(),lr=.01)
p2_best=float('inf');p2_history=[]
for epoch in range(20):
    p2_opt.zero_grad();loss=torch.nn.functional.cross_entropy(p2_head(p2_fx),p2_targets);loss.backward();p2_opt.step()
    with torch.no_grad():vl=float(torch.nn.functional.cross_entropy(p2_head(p2_fv),torch.tensor(valid.label.to_numpy(dtype='int64'))))
    p2_history.append(vl)
    if vl<p2_best:p2_best=vl;p2_state=copy.deepcopy(p2_head.state_dict())
p2_head.load_state_dict(p2_state)
with torch.no_grad():p2_prediction=p2_head(p2_features(test)).argmax(1).numpy()
p2_report=pd.DataFrame([{'method':'frozen MiniLM CLS + linear head','test_macro_f1':f1_score(test.label,p2_prediction,average='macro'),'training_and_evaluation_seconds':time.perf_counter()-p2_start},{'method':'earlier TF-IDF baseline','test_macro_f1':f1_score(test.label,baseline.predict(test.text),average='macro'),'training_and_evaluation_seconds':np.nan}])
display(p2_report)
print({'selected_epoch':int(np.argmin(p2_history))+1,'cached_train_representation_bytes':p2_fx.numel()*p2_fx.element_size(),'actual_trained_head_parameters':sum(p.numel() for p in p2_head.parameters())})
assert len(p2_prediction)==len(test) and np.isfinite(p2_history).all()

### Read the comparison honestly

This is feature extraction with a trained linear head, not end-to-end fine-tuning. The backbone's classifier and pooler are bypassed, so their initialization warnings do not describe the final trained head. CLS vectors are not necessarily the best sentence representation: masked mean pooling is a concrete next comparison. Pretrained familiarity can help, but an exact-vocabulary TF-IDF baseline can be strong on these short authored tickets.

**Try:** prepend 100 irrelevant words to each validation ticket. **Worked reasoning:** truncation may remove the actual request while every tensor shape remains valid. Inspect token coverage, compare a larger token budget or tail-preserving preprocessing on development data, and measure the added encoding cost. Do not repeatedly tune against the test results displayed here.